# Camada Bronze

Este notebook realiza a ingestão dos dados brutos do projeto.

Os arquivos CSV são carregados sem alterações de conteúdo ou estrutura e persistidos em tabelas Delta na camada Bronze. Em cada ingestão é adicionada a coluna `ingestion_datetime`, permitindo identificar quando cada registro entrou no Lakehouse.

Além dos arquivos CSV, este notebook realiza a ingestão da cotação do dólar disponibilizada pela API do Banco Central.

In [0]:
# Configurações compartilhadas do ambiente

from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "olist_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

# Garante que a Landing Zone exista antes da leitura dos arquivos
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.Landing")

DataFrame[]

## Ingestão dos arquivos CSV

A seguir, cada arquivo da Landing Zone é lido individualmente. Nesta etapa nenhuma limpeza ou transformação é aplicada, preservando os dados conforme recebidos na origem.

O schema e uma amostra dos registros são exibidos para permitir a validação da ingestão.

In [0]:
df_info = spark.read.csv(
f"{landing_path}/movies_info_TMDB_IMDB.csv", header=True, inferSchema=True
)
df_info.printSchema()
display(df_info.limit(10))
print("Linhas lidas:", df_info.count())

root
 |-- id: integer (nullable = true)
 |-- tconst: string (nullable = true)
 |-- title: string (nullable = true)
 |-- original_title: string (nullable = true)
 |-- original_language: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- runtime: string (nullable = true)
 |-- status: string (nullable = true)
 |-- overview: string (nullable = true)
 |-- tagline: string (nullable = true)



id,tconst,title,original_title,original_language,release_date,runtime,status,overview,tagline
293660,tt1431045,Deadpool,Deadpool,en,2016-02-09,108,Released,"The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.",Witness the beginning of a happy ending.
299536,tt4154756,AVENGERS: INFINITY WAR,Avengers: Infinity War,en,04-25-2018,149,Released,"As the Avengers and their allies have continued to protect the world from threats too large for any one hero to handle, a new danger has emerged from the cosmic shadows: Thanos. A despot of intergalactic infamy, his goal is to collect all six Infinity Stones, artifacts of unimaginable power, and use them to inflict his twisted will on all of reality. Everything the Avengers have fought for has led up to this moment - the fate of Earth and existence itself has never been more uncertain.",An entire universe. Once and for all.
299534,tt4154796,Avengers: Endgame,Avengers: Endgame,en,2019-04-24,181,released,"After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no matter what consequences may be in store.",Avenge the fallen.
475557,tt7286456,Joker,Joker,en,2019-10-01,122,Released,"During the 1980s, a failed stand-up comedian is driven insane and turns to a life of crime and chaos in Gotham City while becoming an infamous psychopathic crime figure.",Put on a happy face.
271110,tt3498820,Captain America: Civil War,Captain America: Civil War,en,2016-04-27,147,Released,"Following the events of Age of Ultron, the collective governments of the world pass an act designed to regulate all superhuman activity. This polarizes opinion amongst the Avengers, causing two factions to side with Iron Man or Captain America, which causes an epic battle between former allies.",United we stand. Divided we fall.
284054,tt1825683,Black Panther,Black Panther,en,2018-02-13,135,Released,"King T'Challa returns home to the reclusive, technologically advanced African nation of Wakanda to serve as his country's new leader. However, T'Challa soon finds that he is challenged for the throne by factions within his own country as well as without. Using powers reserved to Wakandan kings, T'Challa assumes the Black Panther mantle to join with ex-girlfriend Nakia, the queen-mother, his princess-kid sister, members of the Dora Milaje (the Wakandan 'special forces') and an American secret agent, to prevent Wakanda from being dragged into a world war.",null
284052,tt1211837,Doctor Strange,Doctor Strange,en,2016-10-25,115,Released,"After his career is destroyed, a brilliant but arrogant surgeon gets a new lease on life when a sorcerer takes him under her wing and trains him to defend the world against evil.",The impossibilities are endless.
315635,tt2250912,Spider-Man: Homecoming,Spider-Man: Homecoming,en,2017-07-05,133,RELEASED,"Following the events of Captain America: Civil War, Peter Parker, with the help of his mentor Tony Stark, tries to balance his life as an ordinary high school student in Queens, New York City, with fighting crime as his superhero alter ego Spider-Man as a new threat, the Vulture, emerges.",Homework can wait. The city can't.
283995,tt3896198,Guardians of the Galaxy Vol. 2,Guardians of the Galaxy Vol. 2,en,2017-04-19,137,Released,The Guardians must fight to keep their newfound family together as they unravel the mysteries of Peter Quill's true parentage.,Obviously.
297761,tt1386697,Suicide Squad,Suicide Squad,en,2016-08-03,123,Released,"From DC Comics comes the Suicide Squad, an antihero team of incarcerated sup

Linhas lidas: 106930


In [0]:
df_financeiro = spark.read.csv(
    f"{landing_path}/movies_financials_IMDB_TMDB.csv", header=True, inferSchema=True
)
df_financeiro.printSchema()
display(df_financeiro.limit(10))
print("Linhas lidas:", df_financeiro.count())

root
 |-- id: integer (nullable = true)
 |-- budget: string (nullable = true)
 |-- revenue: string (nullable = true)



id,budget,revenue
293660,58000000,Unknown
299536,300000000,2052415039
299534,356000000,2800000000
475557,55000000,1074458282
271110,250000000,Não Informado
284054,200000000,1349926083
284052,180000000,676343174
315635,175000000,880166924
283995,200000000,863756051
297761,175000000,746846894


Linhas lidas: 106165


In [0]:
df_metricas = spark.read.csv(
    f"{landing_path}/movies_metrics_IMDB_TMDB.csv", header=True, inferSchema=True
)
df_metricas.printSchema()
display(df_metricas.limit(10))
print("Linhas lidas:", df_metricas.count())

root
 |-- id: integer (nullable = true)
 |-- popularity: string (nullable = true)
 |-- vote_average: string (nullable = true)
 |-- vote_count: string (nullable = true)
 |-- averageRating: string (nullable = true)
 |-- numVotes: string (nullable = true)



id,popularity,vote_average,vote_count,averageRating,numVotes
293660,72.735,7.606,28894,8.0,1270339
299536,"154,34",8.255,27713,8.4,1406782
299534,91.756,8.263,23857,8.4,1484150
475557,"54,522",8.168,23425,8.3,1723035
271110,70.741,7.4,21541,7.8,947222
284054,43.665,7.39,null,7.3,924922
284052,70.535,7.427,20935,7.5,895880
315635,65.88,7.345,20507,7.4,835116
283995,67.553,7.624,20353,7.6,844767
297761,35.356,5.909,20097,5.9,null


Linhas lidas: 107364


In [0]:
df_creditos = spark.read.csv(
    f"{landing_path}/credits_and_tags_IMDB_TMDB.csv", header=True, inferSchema=True
)
df_creditos.printSchema()
display(df_creditos.limit(10))
print("Linhas lidas:", df_creditos.count())

root
 |-- id: integer (nullable = true)
 |-- genres: string (nullable = true)
 |-- production_companies: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- spoken_languages: string (nullable = true)
 |-- keywords: string (nullable = true)
 |-- directors: string (nullable = true)
 |-- writers: string (nullable = true)
 |-- cast: string (nullable = true)



id,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,cast
293660,"Action, Adventure, Comedy","20th Century Fox, The Donners' Company, Genre Films",United States of America,English,"superhero, anti hero, mercenary, based on comic, aftercreditsstinger, duringcreditsstinger",Tim Miller,"Rhett Reese, Paul Wernick","Ryan Reynolds, Morena Baccarin, Ed Skrein, T.J. Miller, Gina Carano, Leslie Uggams, Brianna Hildebrand, Stefan Kapičić, Karan Soni, Randal Reeder"
299536,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"English, Xhosa","sacrifice, magic, superhero, based on comic, space, battlefield, genocide, magical object, super power, aftercreditsstinger, marvel cinematic universe (mcu), cosmic","Anthony Russo, Joe Russo",N/A,"Robert Downey Jr., Chris Evans, Chris Hemsworth, Josh Brolin, Mark Ruffalo, Scarlett Johansson, Don Cheadle, Benedict Cumberbatch, Tom Holland, Chadwick Boseman"
299534,"Adventure, Science Fiction, Action",Marvel Studios,United States of America,"English, Japanese, Xhosa","superhero, time travel, space travel, time machine, based on comic, sequel, alien invasion, superhero team, marvel cinematic universe (mcu), alternate timeline, father daughter relationship, sister sister relationship","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Stan Lee, Jack Kirby, Joe Simon, Steve Englehart, Steve Gan, Bill Mantlo, Keith Giffen, Jim Starlin, Larry Lieber, Don Heck","Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner, Josh Brolin, Don Cheadle, Paul Rudd, Benedict Cumberbatch"
475557,"Crime, Thriller, Drama","Warner Bros. Pictures, Joint Effort, Village Roadshow Pictures, Bron Studios, DC Films","Canada, United States of America",English,"dream, street gang, society, psychopath, clown, villain, based on comic, murder, psychological thriller, criminal mastermind, mental illness, anarchy, character study, clown makeup, subway train, social realism, supervillain, tv host, 1980s, mother son relationship, origin story, falling into madness, depressing",Todd Phillips,"Todd Phillips, Scott Silver, Bob Kane, Bill Finger, Jerry Robinson","Joaquin Phoenix, Robert De Niro, Zazie Beetz, Frances Conroy, Brett Cullen, Shea Whigham, Bill Camp, Glenn Fleshler, Leigh Gill, Josh Pais"
271110,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"Romanian, English, German, Russian","civil war, superhero, based on comic, sequel, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu), excited","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Joe Simon, Jack Kirby","Chris Evans, Robert Downey Jr., Scarlett Johansson, Sebastian Stan, Anthony Mackie, Don Cheadle, Jeremy Renner, Chadwick Boseman, Paul Bettany, Elizabeth Olsen"
284054,"Action, Adventure, Science Fiction",Marvel Studios,United States of America,"English, Korean, Swahili, Xhosa","africa, superhero, based on comic, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu)",Ryan Coogler,"Ryan Coogler, Joe Robert Cole, Stan Lee, Jack Kirby","Chadwick Boseman, Michael B. Jordan, Lupita Nyong'o, Danai Gurira, Martin Freeman, Daniel Kaluuya, Letitia Wright, Winston Duke, Sterling K. Brown, Angela Bassett"
284052,"Action, Adventure, Fantasy",Marvel Studios,UNITED STATES OF AMERICA,English,"magic, superhero, training, time, based on comic, sorcerer, doctor, neurosurgeon, wizard, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu)",Scott Derrickson,N/A,"Benedict Cumberbatch, Chiwetel Ejiofor, Rachel McAdams, Benedict Wong, Mads Mikkelsen, Tilda Swinton, Michael Stuhlbarg, Benjamin Bratt, Scott Adkins, Zara Phythian"
315635,"Action, Adventure, Science Fiction, Drama","Marvel Studios, Pascal Pictures, LStar Capital, Columbia Pictures",United States of America,English,"high school, new york city, washington dc, usa, superhero, based on comic, reboot, aftercredits

Linhas lidas: 106320


In [0]:
df_avaliacoes = spark.read.csv(
    f"{landing_path}/movies_reviews.csv", header=True, inferSchema=True
)
df_avaliacoes.printSchema()
display(df_avaliacoes.limit(10))
print("Linhas lidas:", df_avaliacoes.count())

root
 |-- id: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- nota: double (nullable = true)
 |-- comentario: string (nullable = true)



id,nome,nota,comentario
442113,Mariana Cardoso 277,4.4,null
637007,Lucas Reis 602,3.9,null
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.
413036,Gabriela Monteiro 401,7.5,null
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.
387727,Maria Alves 707,8.2,"Gostei bastante, recomendo."
1032506,Amanda Castro 242,4.7,"Fraco, não recomendo."
446554,Sandra Rodrigues 736,6.5,Assisti até o final mas não me marcou.
569916,Camila Lima 489,8.2,Muito bom! Vale a pena assistir.
864552,Roberto Almeida 555,9.3,"Obra-prima do cinema, simplesmente espetacular."


Linhas lidas: 32412


## Persistência das tabelas Bronze

Os DataFrames carregados são gravados em formato Delta utilizando `append`.

Antes da gravação é adicionada a coluna `ingestion_datetime`, contendo o momento da ingestão. Essa informação será utilizada posteriormente na Silver para identificar a versão mais recente de registros duplicados.

In [0]:
# A Bronze preserva os dados da origem e adiciona apenas o timestamp de ingestão.
# O modo append mantém o histórico das diferentes execuções do pipeline.
(df_info
     .withColumn("ingestion_datetime", F.current_timestamp())
     .write.format("delta").mode("append")
     .saveAsTable(f"{catalog}.bronze.tb_movies_info"))

(df_financeiro
     .withColumn("ingestion_datetime", F.current_timestamp())
     .write.format("delta").mode("append")
     .saveAsTable(f"{catalog}.bronze.tb_movies_financials"))

(df_metricas
     .withColumn("ingestion_datetime", F.current_timestamp())
     .write.format("delta").mode("append")
     .saveAsTable(f"{catalog}.bronze.tb_movies_metrics"))

(df_creditos
     .withColumn("ingestion_datetime", F.current_timestamp())
     .write.format("delta").mode("append")
     .saveAsTable(f"{catalog}.bronze.tb_credits_and_tags"))

(df_avaliacoes
     .withColumn("ingestion_datetime", F.current_timestamp())
     .write.format("delta").mode("append")
     .saveAsTable(f"{catalog}.bronze.tb_movies_reviews"))
 


## Ingestão da cotação do dólar — Banco Central

A atividade exige a obtenção da cotação do dólar pela API PTAX do Banco Central.

As datas são recebidas através de widgets do Databricks. Por padrão, o notebook consulta os últimos sete dias corridos, pois finais de semana e feriados podem não possuir cotação.

In [0]:
from datetime import date, timedelta, datetime

# Datas padrão: últimos 7 dias
hoje = date.today()
data_inicial_padrao = hoje - timedelta(days=7)

# Cria os widgets.
# O Databricks mantém valores já preenchidos caso os widgets existam.
dbutils.widgets.text(
    "data_inicial",
    data_inicial_padrao.strftime("%Y-%m-%d"),
    "Data inicial"
)

dbutils.widgets.text(
    "data_final",
    hoje.strftime("%Y-%m-%d"),
    "Data final"
)

# Lê os valores dos widgets
data_inicial_texto = dbutils.widgets.get("data_inicial")
data_final_texto = dbutils.widgets.get("data_final")

# Converte para objetos date
data_inicial = datetime.strptime(
    data_inicial_texto,
    "%Y-%m-%d"
).date()

data_final = datetime.strptime(
    data_final_texto,
    "%Y-%m-%d"
).date()

# A API do BCB espera MM-DD-YYYY
data_inicio_formatada = data_inicial.strftime("%m-%d-%Y")
data_fim_formatada = data_final.strftime("%m-%d-%Y")

print("Período selecionado:")
print("Widget:", data_inicial_texto, "até", data_final_texto)
print("BCB:", data_inicio_formatada, "até", data_fim_formatada)

Período selecionado:
Widget: 2026-10-07 até 2026-09-30
BCB: 10-07-2026 até 09-30-2026


### Consulta à API

É definido explicitamente o schema esperado da resposta para evitar inferência incorreta de tipos.

Primeiramente é realizada a requisição direta à API. Caso o ambiente do Databricks não consiga acessar o serviço externo, é utilizado um arquivo JSON contendo uma resposta da mesma API como fallback, permitindo a continuidade da atividade.

In [0]:
import requests
import json
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    ArrayType,
    StringType,
    DoubleType
)

# Endpoint da API do Banco Central
endpoint = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
)

params = {
    "@dataInicial": f"'{data_inicio_formatada}'",
    "@dataFinalCotacao": f"'{data_fim_formatada}'",
    "$select": "dataHoraCotacao,cotacaoCompra",
    "$format": "json"
}

# Schema da resposta JSON
schema_json = StructType([
    StructField(
        "value",
        ArrayType(
            StructType([
                StructField(
                    "dataHoraCotacao",
                    StringType(),
                    True
                ),
                StructField(
                    "cotacaoCompra",
                    DoubleType(),
                    True
                )
            ])
        ),
        True
    )
])

try:
    # Tenta consultar diretamente a API
    resposta = requests.get(
        endpoint,
        params=params,
        timeout=30
    )

    resposta.raise_for_status()

    dados = resposta.json()

    print("API do BCB consultada com sucesso.")
    print("URL:", resposta.url)
    print(
        "Quantidade de registros:",
        len(dados.get("value", []))
    )

    # Converte a resposta para DataFrame Spark
    df_json_bcb = (
        spark.createDataFrame(
            [(json.dumps(dados),)],
            ["json"]
        )
        .select(
            F.from_json(
                F.col("json"),
                schema_json
            ).alias("dados")
        )
        .select("dados.*")
    )

# Fallback utilizado apenas quando o ambiente Databricks não consegue
# acessar o endpoint externo do Banco Central.
except Exception as e:

    print("Não foi possível acessar a API do BCB.")
    print("Erro:", e)
    print("Utilizando arquivo JSON de fallback.")

    # O arquivo deve estar dentro do Volume Landing
    caminho_json = (
        "/Volumes/olist_medallion/"
        "bronze/Landing/cotacao_bcb.json"
    )

    df_json_bcb = (
        spark.read
        .option("multiline", "true")
        .schema(schema_json)
        .json(caminho_json)
    )

Não foi possível acessar a API do BCB.
Erro: HTTPSConnectionPool(host='olinda.bcb.gov.br', port=443): Max retries exceeded with url: /olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?%40dataInicial=%2710-07-2026%27&%40dataFinalCotacao=%2709-30-2026%27&%24select=dataHoraCotacao%2CcotacaoCompra&%24format=json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7fc46e483b30>: Failed to resolve 'olinda.bcb.gov.br' ([Errno -3] Temporary failure in name resolution)"))
Utilizando arquivo JSON de fallback.


 A extração acima foi implementada conforme especificado na atividade, entretanto durante a execução no databricks, ocorreu um erro de resolução de DNS ao acessar o domínio da API do Banco Central. Como alternativa, realizei a requisição localmente salvando seu retorno em formato JSON e enviei para o databricks


### Preparação da resposta da API

A resposta da API possui os registros de cotação dentro do campo `value`. O array é explodido para que cada cotação corresponda a uma linha.

Nesta etapa também são selecionados apenas os campos necessários e adicionada a coluna `ingestion_datetime`, seguindo o mesmo padrão das demais tabelas Bronze.

In [0]:
df_cotacao_dolar = (
    df_json_bcb

    # A resposta do BCB possui os registros dentro do array "value"
    .select(
        F.explode("value").alias("cotacao")
    )

    # Seleciona somente os campos necessários
    .select(
        F.to_timestamp(
            F.col("cotacao.dataHoraCotacao")
        ).alias("dataHoraCotacao"),

        F.col("cotacao.cotacaoCompra")
        .cast("decimal(18,6)")
        .alias("cotacaoCompra")
    )

    # Timestamp da ingestão na camada Bronze
    .withColumn(
        "ingestion_datetime",
        F.current_timestamp()
    )
)

display(df_cotacao_dolar)

dataHoraCotacao,cotacaoCompra,ingestion_datetime
2023-12-18T13:05:33.405Z,null,2026-10-07T15:16:13.585Z
2023-12-19T13:04:31.068Z,null,2026-10-07T15:16:13.585Z
2023-12-20T13:04:28.828Z,null,2026-10-07T15:16:13.585Z
2023-12-21T13:07:35.231Z,null,2026-10-07T15:16:13.585Z
2023-12-22T13:02:59.214Z,null,2026-10-07T15:16:13.585Z
2023-12-26T13:07:41.053Z,null,2026-10-07T15:16:13.585Z
2023-12-27T13:05:01.784Z,null,2026-10-07T15:16:13.585Z
2023-12-28T13:04:46.986Z,null,2026-10-07T15:16:13.585Z
2023-12-29T10:04:11.873Z,null,2026-10-07T15:16:13.585Z
2024-01-02T13:05:50.319Z,null,2026-10-07T15:16:13.585Z


### Persistência da cotação na Bronze

Os dados obtidos são gravados em formato Delta e no modo `append`, mantendo o histórico de ingestões da cotação.

In [0]:
(
    df_cotacao_dolar.write
    .format("delta")
    .mode("append")
    .saveAsTable(
        f"{catalog}.bronze.tb_cotacao_dolar"
    )
)

print("Cotação salva na Bronze com sucesso.")

Cotação salva na Bronze com sucesso.


## Validação da ingestão

As verificações abaixo confirmam que os registros foram persistidos na tabela Bronze e permitem conferir o intervalo de datas e a quantidade de cotações obtidas.

In [0]:
df_verificacao = spark.table(
    f"{catalog}.bronze.tb_cotacao_dolar"
)

display(
    df_verificacao
    .select(
        F.to_date("dataHoraCotacao").alias("data_cotacao"),
        "cotacaoCompra",
        "ingestion_datetime"
    )
    .orderBy(
        F.col("data_cotacao").desc()
    )
)

data_cotacao,cotacaoCompra,ingestion_datetime
2024-12-31,6.191700,2026-10-07T02:55:28.956Z
2024-12-31,6.191700,2026-10-07T07:09:43.657Z
2024-12-31,null,2026-10-07T16:13:13.626Z
2024-12-31,6.191700,2026-10-07T02:20:22.983Z
2024-12-31,null,2026-10-07T15:17:02.432Z
2024-12-30,6.191700,2026-10-07T02:20:22.983Z
2024-12-30,null,2026-10-07T15:17:02.432Z
2024-12-30,6.191700,2026-10-07T02:55:28.956Z
2024-12-30,null,2026-10-07T16:13:13.626Z
2024-12-30,6.191700,2026-10-07T07:09:43.657Z


In [0]:
df_verificacao.select(
    F.min(
        F.to_date("dataHoraCotacao")
    ).alias("menor_data"),

    F.max(
        F.to_date("dataHoraCotacao")
    ).alias("maior_data"),

    F.count("*").alias("quantidade")
).show()

+----------+----------+----------+
|menor_data|maior_data|quantidade|
+----------+----------+----------+
|2023-12-18|2024-12-31|      1310|
+----------+----------+----------+



In [0]:
print("Quantidade:", df_cotacao_dolar.count())

df_cotacao_dolar.select(
    F.min("dataHoraCotacao").alias("menor_data"),
    F.max("dataHoraCotacao").alias("maior_data")
).show(truncate=False)

Quantidade: 262
+-----------------------+-----------------------+
|menor_data             |maior_data             |
+-----------------------+-----------------------+
|2023-12-18 13:05:33.405|2024-12-31 10:04:09.264|
+-----------------------+-----------------------+

